# Full pipeline: from the committed weights to the submission

This notebook reproduces both submissions of the
[Solar Filament Segmentation Challenge 2026](https://www.kaggle.com/competitions/filament-segmentation-2026)
from the weights stored in this repository, and shows what the model does on
frames it did not see in training.

| step | what | where in the package |
|---|---|---|
| model | HRNet-W32 U-Net, 1024 × 1024 input, five models from a five-fold cross-validation | `weights/`, `filament.pipeline.load_fold_models` |
| test-time augmentation | each model's probability averaged over the eight flips and quarter turns, then the five models averaged | `filament.pipeline.predict_ensemble_tta` |
| post-processing | map upsampled to 2048, threshold 0.40, fragments rejoined across gaps of up to 16 px (at 1024), regions under 400 px dropped, regions whose mean probability is under 0.65 dropped | `filament.pipeline.FINAL_SETTING` |
| output | one compressed COCO RLE per filament, checked that no two masks of a frame share a pixel | `filament.pipeline.submission_from_maps` |

Two files are written from the same probability maps:

- `submission.csv` — the final configuration (pooled five-fold PQ 0.4127, public leaderboard 0.36);
- `submission_no_score_cut.csv` — the same without the cut on mean probability (0.4065, public leaderboard 0.36).

The experiments behind each choice are in `docs/experiments/`.

**On Kaggle:** attach the competition data, turn on the internet (to clone the
repository) and select a GPU (one T4 is enough). **Locally:** run from
`notebooks/` inside a clone with `uv sync`; the data is found through
`configs/paths.yaml` or the `MAGFILO_ROOT` environment variable.

## 1. The code and the weights

On Kaggle the repository is cloned, with its `weights/` directory, into `/tmp`
so that it does not count against the notebook's output files. Only the two
packages Kaggle lacks are installed, at the versions pinned in
`requirements.txt`; Kaggle's own CUDA build of torch is kept.

In [ ]:
import sys
import time
from pathlib import Path

NOTEBOOK_STARTED = time.perf_counter()

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "main"  # branch, tag or commit hash
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    REPO = Path("/tmp/repo")
    !rm -rf {REPO}
    !git clone -q --depth 1 --branch {REF} {REPO_URL} {REPO}
    !pip install -q "segmentation-models-pytorch==0.5.0" "timm==1.0.29"
else:
    REPO = Path.cwd().resolve().parent
!cd {REPO} && git log --oneline -1

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("filament", filament.__version__, "| smp", smp.__version__, "| torch", torch.__version__)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 2. The data

The competition data is found by its annotation file, wherever Kaggle mounts it.

In [ ]:
import os

from filament.paths import load_paths

if ON_KAGGLE:
    found = sorted(Path("/kaggle/input").glob("**/MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not found:
        raise SystemExit("Attach the competition data to this notebook first.")
    os.environ["MAGFILO_ROOT"] = str(found[0].parent.parent)

paths = load_paths().require_dataset()
test_frames = sorted(paths.test_images.glob("*.jpeg"))
print("train frames:", len(list(paths.train_images.glob("*.jpeg"))))
print("test frames: ", len(test_frames))

## 3. Load the five fold models

Each file holds the weights in half precision, to stay under GitHub's 100 MB
limit per file, together with the configuration the model was trained with.
They are restored to single precision when loaded. The fold and resolution
stored in each file are checked against its name.

In [ ]:
from filament.pipeline import FOLDS, load_fold_models

started = time.perf_counter()
models = load_fold_models(REPO / "weights", device=DEVICE)
load_seconds = time.perf_counter() - started
parameters = sum(parameter.numel() for parameter in models[0].parameters())
print(f"{len(models)} models, {parameters / 1e6:.1f} M parameters each, loaded in {load_seconds:.1f} s")

## 4. Predict the test frames

Each test frame is passed through all five models in all eight views, 40
forward passes per frame; the 40 probability maps are averaged.

In [ ]:
import logging

from filament.pipeline import predict_maps

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")

started = time.perf_counter()
test_maps = predict_maps(models, test_frames, device=DEVICE)
inference_seconds = time.perf_counter() - started
print(
    f"{len(test_maps)} frames in {inference_seconds:.0f} s "
    f"({inference_seconds / len(test_maps):.2f} s per frame, 40 forward passes each)"
)

## 5. Write the two submissions

Both are checked for overlapping masks before they are reported as written:
Kaggle rejects a file in which two masks of one frame share a pixel.

In [ ]:
from filament.pipeline import FINAL_SETTING, NO_SCORE_CUT_SETTING, submission_from_maps

OUTPUT = Path("/kaggle/working") if ON_KAGGLE else REPO / "outputs" / "full_pipeline"
OUTPUT.mkdir(parents=True, exist_ok=True)

started = time.perf_counter()
submissions = {}
for name, setting in (
    ("submission.csv", FINAL_SETTING),
    ("submission_no_score_cut.csv", NO_SCORE_CUT_SETTING),
):
    rows = submission_from_maps(test_maps, paths.test_images, setting, OUTPUT / name)
    submissions[name] = rows
    frames = rows["filament_id"].str.rsplit("_", n=1).str[0].nunique()
    print(f"{name}: {len(rows)} filaments on {frames} of {len(test_maps)} frames")
postprocess_seconds = time.perf_counter() - started
print(f"post-processing both: {postprocess_seconds:.0f} s")
submissions["submission.csv"].head()

## 6. Check the weights on frames they did not see

The fold-0 model is run on the 142 frames held out of its training, in the
eight views, and scored with the final post-processing against every
annotator. This should give the fold-0 PQ recorded in the experiments
(0.4016, from single-precision weights): a much lower number would mean the
weights or the code in this repository are not the ones that made the
submission.

PQ counts a predicted filament as found when its IoU with an annotated one
exceeds 0.5, and is the product of two parts: SQ, the mean IoU of those
matches, and RQ, an F1 score of the detections. Each frame is scored once per
annotator who labelled it.

In [ ]:
from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.metrics.pq import compute_pq
from filament.submit.rle import masks_to_gt_df

FOLD = 0
RECORDED_FOLD0_PQ = 0.4016

dataset = load_annotations(paths.train_annotations)
held_out = sorted(load_fold(FOLD, REPO / "configs" / "splits").val)
held_out_frames = [paths.train_images / f"{stem}.jpeg" for stem in held_out]

started = time.perf_counter()
val_maps = predict_maps([models[FOLD]], held_out_frames, device=DEVICE)
val_rows = submission_from_maps(
    val_maps, paths.train_images, FINAL_SETTING, OUTPUT / "fold0_predictions.csv"
)
truth = masks_to_gt_df(dataset, held_out)
fold0 = compute_pq(truth, val_rows)
print(f"fold 0, {len(held_out)} frames, {time.perf_counter() - started:.0f} s")
print(fold0)
print(f"recorded: {RECORDED_FOLD0_PQ}, difference {fold0.pq - RECORDED_FOLD0_PQ:+.4f}")

### Quality of the matched filaments

For every annotated filament that was found, the IoU and Dice of the match.
SQ is the mean of the IoU values; the cut at 0.5 is what makes a match.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

figure, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for axis, values, label in (
    (axes[0], fold0.iou_scores, "IoU"),
    (axes[1], fold0.dice_scores, "Dice"),
):
    axis.hist(values, bins=np.linspace(0.5 if label == "IoU" else 0.66, 1, 26), color="#4C72B0")
    axis.axvline(np.mean(values), color="black", linestyle="--", linewidth=1)
    axis.set_title(f"{label} of the {len(values)} matches (mean {np.mean(values):.3f})")
    axis.set_xlabel(label)
axes[0].set_ylabel("matches")
plt.tight_layout()
plt.show()

### What the predictions look like

Held-out frames of fold 0, with the annotations of the first annotator in
green and the predicted filaments in magenta, each filament its own outline.
The frames are chosen at even steps through the sorted list, not by score.

In [ ]:
import cv2

from filament.data.image import load_grayscale
from filament.submit.rle import rle_to_mask


def outlines(image, masks, colour):
    # Drawn at 2048 so that thin filaments keep a visible outline.
    for mask in masks:
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(image, contours, -1, colour, 4)
    return image


def predicted_masks(rows, stem):
    chosen = rows[rows["filament_id"].str.rsplit("_", n=1).str[0] == stem]
    return [rle_to_mask(counts) for counts in chosen["segmentation_rle"]]


def annotated_masks(stem):
    entry = next(entry for entry in dataset.annotator_images if entry.image_id.endswith(f"-{stem}"))
    return [annotation.to_mask(entry.height, entry.width) for annotation in entry.annotations]


examples = held_out[:: len(held_out) // 4][:4]
figure, axes = plt.subplots(1, 4, figsize=(20, 5.4))
for axis, stem in zip(axes, examples):
    frame = cv2.cvtColor(load_grayscale(paths.train_images / f"{stem}.jpeg"), cv2.COLOR_GRAY2RGB)
    outlines(frame, annotated_masks(stem), (0, 200, 0))
    outlines(frame, predicted_masks(val_rows, stem), (230, 0, 230))
    axis.imshow(frame)
    axis.set_title(stem)
    axis.axis("off")
plt.tight_layout()
plt.show()

And four test frames with the filaments of `submission.csv` (no annotations
are available for them).

In [ ]:
test_stems = sorted(test_maps)
figure, axes = plt.subplots(1, 4, figsize=(20, 5.4))
for axis, stem in zip(axes, test_stems[:: len(test_stems) // 4][:4]):
    frame = cv2.cvtColor(load_grayscale(paths.test_images / f"{stem}.jpeg"), cv2.COLOR_GRAY2RGB)
    outlines(frame, predicted_masks(submissions["submission.csv"], stem), (230, 0, 230))
    axis.imshow(frame)
    axis.set_title(stem)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. Cost

In [ ]:
print(f"loading the five models:      {load_seconds:6.1f} s")
print(
    f"test inference, {len(test_maps)} frames:   {inference_seconds:6.1f} s "
    f"({inference_seconds / len(test_maps):.2f} s per frame)"
)
print(f"post-processing, both files:  {postprocess_seconds:6.1f} s")
print(f"whole notebook:               {time.perf_counter() - NOTEBOOK_STARTED:6.1f} s")
print("written:", ", ".join(sorted(path.name for path in OUTPUT.glob("*.csv"))))